# Deepfake Detection — Improved Xception Training

This is a separate notebook; it does not modify the original project. It improves the training workflow with stronger augmentation, controlled fine-tuning, validation-AUC checkpointing, and validation-based threshold selection.

## Before running

Update the three dataset paths in the next cell. The folders must contain `Fake` and `Real` subfolders. Keep the test set untouched until final evaluation.

In [1]:
import os
import random
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS_STAGE1 = 10
EPOCHS_STAGE2 = 15

# Change these only if your dataset is stored elsewhere.
train_dir = r'C:\MTech_Project_Deepfake_Detection\dataset\Balanced_Binary_Dataset\train'
validation_dir = r'C:\MTech_Project_Deepfake_Detection\dataset\Balanced_Binary_Dataset\validation'
test_dir = r'C:\MTech_Project_Deepfake_Detection\dataset\Balanced_Binary_Dataset\test'

for folder in (train_dir, validation_dir, test_dir):
    assert os.path.isdir(folder), f'Dataset folder not found: {folder}'
print('TensorFlow:', tf.__version__)

TensorFlow: 2.21.0


In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.xception import preprocess_input

# These transformations simulate common resizing, lighting, and compression-related image changes.
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=12,
    width_shift_range=0.12,
    height_shift_range=0.12,
    zoom_range=0.15,
    brightness_range=(0.80, 1.20),
    horizontal_flip=True,
    fill_mode='nearest'
)
eval_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=True, seed=SEED
)
validation_generator = eval_datagen.flow_from_directory(
    validation_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)
test_generator = eval_datagen.flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)

print('Class indices:', train_generator.class_indices)
assert train_generator.class_indices == {'fake': 0, 'real': 1}, 'Expected folders named fake and real'

Found 3718 images belonging to 2 classes.
Found 798 images belonging to 2 classes.
Found 794 images belonging to 2 classes.
Class indices: {'fake': 0, 'real': 1}


In [3]:
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

base_model = Xception(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = BatchNormalization()(x)
x = Dense(512, activation='relu', kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.45)(x)
x = Dense(256, activation='relu', kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.30)(x)
output = Dense(1, activation='sigmoid', name='real_probability')(x)
model = Model(inputs=base_model.input, outputs=output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1_conv1 (Conv2D)         │ (None, 111, 111, 32)      │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1_conv1_bn               │ (None, 111, 111, 32)      │             128 │ block1_conv1[0][0]         │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1_conv1_act (Activation) │ (None, 111, 111, 32)      │               0 │ block1_conv1_bn[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1_conv2 (Conv2D)         │ (None, 109, 109, 64)      │          18,432 │ block1_conv1_act[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1_conv2_bn               │ (None, 109, 109, 64)      │             256 │ block1_conv2[0][0]         │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1_conv2_act (Activation) │ (None, 109, 109, 64)      │               0 │ block1_conv2_bn[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block2_sepconv1               │ (None, 109, 109, 128)     │           8,768 │ block1_conv2_act[0][0]     │
│ (SeparableConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block2_sepconv1_bn            │ (None, 109, 109, 128)     │             512 │ block2_sepconv1[0][0]      │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block2_sepconv2_act           │ (None, 109, 109, 128)     │               0 │ block2_sepconv1_bn[0][0]   │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block2_sepconv2               │ (None, 109, 109, 128)     │          17,536 │ block2_sepconv2_act[0][0]  │
│ (SeparableConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block2_sepconv2_bn            │ (None, 109, 109, 128)     │             512 │ block2_sepconv2[0][0]      │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 55, 55, 128)       │           8,192 │ block1_conv2_act[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block2_pool (MaxPooling2D)    │ (None, 55, 55, 128)       │               

 Total params: 22,050,345 (84.12 MB)

 Trainable params: 1,184,769 (4.52 MB)

 Non-trainable params: 20,865,576 (79.60 MB)

In [4]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import AUC, Precision, Recall
from tensorflow.keras.optimizers import AdamW

MODEL_PATH = 'best_xception_improved.keras'

def compile_model(learning_rate):
    model.compile(
        optimizer=AdamW(learning_rate=learning_rate, weight_decay=1e-5),
        loss=BinaryCrossentropy(label_smoothing=0.03),
        metrics=['accuracy', Precision(name='precision'), Recall(name='recall'), AUC(name='auc')]
    )

def callbacks():
    # All callbacks use validation AUC, avoiding inconsistent save/stop criteria.
    return [
        ModelCheckpoint(MODEL_PATH, monitor='val_auc', mode='max', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.25, patience=2, min_lr=1e-7, verbose=1),
    ]

compile_model(1e-4)
history_stage1 = model.fit(train_generator, validation_data=validation_generator, epochs=EPOCHS_STAGE1, callbacks=callbacks())

Epoch 1/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5387 - auc: 0.5450 - loss: 0.9336 - precision: 0.5409 - recall: 0.5121
Epoch 1: val_auc improved from None to 0.65379, saving model to best_xception_improved.keras

Epoch 1: finished saving model to best_xception_improved.keras
233/233 ━━━━━━━━━━━━━━━━━━━━ 689s 3s/step - accuracy: 0.5387 - auc: 0.5450 - loss: 0.9336 - precision: 0.5409 - recall: 0.5121 - val_accuracy: 0.6065 - val_auc: 0.6538 - val_loss: 0.7811 - val_precision: 0.5873 - val_recall: 0.7168 - learning_rate: 1.0000e-04
Epoch 2/10
233/233 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6011 - auc: 0.6353 - loss: 0.8344 - precision: 0.6022 - recall: 0.5960
Epoch 2: val_auc improved from 0.65379 to 0.71385, saving model to best_xception_improved.keras

Epoch 2: finished saving model to best_xception_improved.keras
233/233 ━━━━━━━━━━━━━━━━━━━━ 703s 3s/step - accuracy: 0.6011 - auc: 0.6353 - loss: 0.8344 - precision: 0.6022 - recall: 0.5960 - val_accuracy: 0.6642 -

In [5]:
# Fine-tune the final Xception blocks. Batch-normalization layers remain frozen for stability.
for layer in base_model.layers[:-80]:
    layer.trainable = False
for layer in base_model.layers[-80:]:
    layer.trainable = not isinstance(layer, tf.keras.layers.BatchNormalization)

print('Trainable base-model layers:', sum(layer.trainable for layer in base_model.layers))
compile_model(5e-6)
history_stage2 = model.fit(train_generator, validation_data=validation_generator, epochs=EPOCHS_STAGE2, callbacks=callbacks())

Trainable base-model layers: 56
Epoch 1/15
233/233 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6869 - auc: 0.7573 - loss: 0.7083 - precision: 0.6956 - recall: 0.6649
Epoch 1: val_auc improved from None to 0.81397, saving model to best_xception_improved.keras

Epoch 1: finished saving model to best_xception_improved.keras
233/233 ━━━━━━━━━━━━━━━━━━━━ 597s 2s/step - accuracy: 0.6869 - auc: 0.7573 - loss: 0.7083 - precision: 0.6956 - recall: 0.6649 - val_accuracy: 0.7444 - val_auc: 0.8140 - val_loss: 0.6605 - val_precision: 0.7893 - val_recall: 0.6667 - learning_rate: 5.0000e-06
Epoch 2/15
233/233 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7453 - auc: 0.8287 - loss: 0.6342 - precision: 0.7585 - recall: 0.7197
Epoch 3: val_auc improved from 0.84416 to 0.86985, saving model to best_xception_improved.keras

Epoch 3: finished saving model to best_xception_improved.keras
233/233 ━━━━━━━━━━━━━━━━━━━━ 643s 3s/step - accuracy: 0.7453 - auc: 0.8287 - loss: 0.6342 - precision: 0.7585 - recall:

In [6]:
# Choose the classification threshold on validation data only — never tune it on the test set.
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

best_model = tf.keras.models.load_model(MODEL_PATH)
validation_generator.reset()
val_prob = best_model.predict(validation_generator, verbose=1).ravel()
val_true = validation_generator.classes

thresholds = np.arange(0.20, 0.81, 0.01)
val_scores = [accuracy_score(val_true, val_prob >= threshold) for threshold in thresholds]
best_threshold = float(thresholds[np.argmax(val_scores)])
print(f'Best validation threshold: {best_threshold:.2f}')
print(f'Validation accuracy at threshold: {max(val_scores):.2%}')

50/50 ━━━━━━━━━━━━━━━━━━━━ 57s 1s/step
Best validation threshold: 0.35
Validation accuracy at threshold: 89.97%


In [7]:
# Final, one-time evaluation on the untouched test set.
test_generator.reset()
test_prob = best_model.predict(test_generator, verbose=1).ravel()
test_true = test_generator.classes
test_pred = (test_prob >= best_threshold).astype(int)

print(f'Test accuracy: {accuracy_score(test_true, test_pred):.2%}')
print(f'Test AUC: {roc_auc_score(test_true, test_prob):.4f}')
print(classification_report(test_true, test_pred, target_names=['Fake', 'Real']))

# Keep these values for the Streamlit frontend.
print(f'Use MODEL_PATH = {MODEL_PATH}')
print(f'Use decision threshold = {best_threshold:.2f}')

50/50 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step
Test accuracy: 89.29%
Test AUC: 0.9631
              precision    recall  f1-score   support

        Fake       0.88      0.91      0.89       397
        Real       0.90      0.88      0.89       397

    accuracy                           0.89       794
   macro avg       0.89      0.89      0.89       794
weighted avg       0.89      0.89      0.89       794

Use MODEL_PATH = best_xception_improved.keras
Use decision threshold = 0.35
